In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
from transformers import ViTImageProcessor, ViTForImageClassification
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Pointing to your standardized 4-class split
BASE_DIR = '/content/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Train_Test_Split_Vit_Data'
train_data_dir = os.path.join(BASE_DIR, 'train')
val_data_dir = os.path.join(BASE_DIR, 'val')

# --- 2. Initialize TransUNet (Hybrid Encoder) ---
# TransUNet typically uses a R50+ViT-Base hybrid as the backbone
model_name = "google/vit-hybrid-base-bit-384"
feature_extractor_transunet = ViTImageProcessor.from_pretrained(model_name)

model_transunet = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=4,
    ignore_mismatched_sizes=True # Re-initializes for your 4 pneumonia classes
)
model_transunet.to(device)

# --- 3. Dataset & Dataloaders ---
class PneumoniaDataset(Dataset):
    def __init__(self, data_dir, feature_extractor):
        self.data_dir = data_dir
        self.feature_extractor = feature_extractor
        self.image_paths = []
        self.labels = []
        self.classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        self.class_to_idx = {name: i for i, name in enumerate(self.classes)}

        for class_name in self.classes:
            path = os.path.join(data_dir, class_name)
            for img in os.listdir(path):
                if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(path, img))
                    self.labels.append(self.class_to_idx[class_name])

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        # TransUNet encoder variants often benefit from the higher 384x384 resolution
        inputs = self.feature_extractor(images=image, return_tensors='pt')
        return inputs['pixel_values'].squeeze(0), torch.tensor(self.labels[idx])

train_dataloader = DataLoader(PneumoniaDataset(train_data_dir, feature_extractor_transunet), batch_size=16, shuffle=True)
val_dataloader = DataLoader(PneumoniaDataset(val_data_dir, feature_extractor_transunet), batch_size=16, shuffle=False)

# --- 4. Optimizer ---
optimizer = AdamW(model_transunet.parameters(), lr=1e-5) # Lower LR for the complex hybrid backbone
criterion = nn.CrossEntropyLoss()

In [ ]:
epochs = 25
print(f"Starting TransUNet-style Hybrid Training on {device}...")

for epoch in range(epochs):
    model_transunet.train()
    train_loss, correct, total = 0, 0, 0
    for batch in train_dataloader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_transunet(inputs).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    # Validation Phase
    model_transunet.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for v_batch in val_dataloader:
            v_inputs, v_labels = v_batch
            v_inputs, v_labels = v_inputs.to(device), v_labels.to(device)
            val_outputs = model_transunet(v_inputs).logits
            _, v_pred = val_outputs.max(1)
            val_total += v_labels.size(0)
            val_correct += v_pred.eq(v_labels).sum().item()

    print(f"TransUNet Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_dataloader):.4f} | "
          f"Train Acc: {100.*correct/total:.2f}% | Val Acc: {100.*val_correct/val_total:.2f}%")

In [ ]:
model_transunet.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_dataloader:
        inputs, labels = batch
        outputs = model_transunet(inputs.to(device)).logits
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Confusion Matrix for the Hybrid_Vit/TransUNet folder
plt.figure(figsize=(10, 8))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='mako', xticklabels=train_dataloader.dataset.classes, yticklabels=train_dataloader.dataset.classes)
plt.title('TransUNet Encoder Confusion Matrix')
plt.show()

print("\nTransUNet Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=train_dataloader.dataset.classes))